# TC-WPN — Stage C: blinding fix + mechanism ablation

**Accelerator: GPU T4.** This covers your supervisor's Phase 1 (fix the blinded
run) and Phase 2 (mechanism ablation). Nothing beyond that: no seed sweep, no
K sweep, no tuning.

### What went wrong in Stage B
`--blind dx_meds` failed twice. That name comes from `src/tcwpn/blinding.py`;
`scripts/tokenize_cohort.py` carries its own vocabulary list with different keys
(`anxiety`, `meds`, `anx_meds`, `psych`). The notebook used the wrong one, so the
blinded pkl was never built and the two evaluate calls found nothing. My error,
and it means **no blinded result exists yet** — do not describe TC-WPN as robust
to lexical shortcuts on the strength of Stage B.

### What this notebook must settle
Your Stage B headline was ProtoNet 0.4928 vs TC-WPN 0.7335, DeLong p ≈ 1.25e-51.
Those two configs differ in **four** ways at once, not one:

| switch | `protonet` | `tcwpn_full` |
|---|---|---|
| temporal weight `w^T` | off | on |
| prototype consistency `w^C` | off | on |
| learnable temperature `tau` | **frozen at 10.0** | learned (went 10.00 -> 6.94) |
| auxiliary CE head | 0.0 | 0.3 |

So +0.241 AUROC is currently attributable to "TC-WPN as a bundle", not to the two
mechanisms the paper is about. `protonet_temp` is the row that decides this, and
it is the first thing this notebook runs.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Repo + deps
# ---------------------------------------------------------------------------
!rm -rf /kaggle/working/tcwpn_test
!git clone -q https://github.com/dulhara79/tcwpn_test.git /kaggle/working/tcwpn_test
%cd /kaggle/working/tcwpn_test
!pip install -q -r requirements.txt 2>&1 | tail -2

import torch, warnings, logging
# Suppress the Bio_ClinicalBERT MLM/NSP head report. Those keys are unexpected
# because AutoModel loads the encoder only; the encoder weights load fine.
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.GradScaler.*")
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 1b. Preflight — fail here, not eight minutes in

Stage C previously died on every training and evaluation call with
`ModuleNotFoundError: No module named 'tcwpn.model'`, but only *after* twelve
minutes of tokenisation had already run. The cause was repository state, not
code: `src/tcwpn/model.py` had been uploaded as `old_model_1.py`, so the module
the pipeline imports did not exist.

This cell runs `tests/test_repo_layout.py`, which checks that every required
module and script is present, that no `_1` / `old_` duplicates are shadowing
them, and that the lambda/beta logging patch survived. It costs under a second.
If it fails, stop and fix the repository before spending any GPU time.

In [ ]:
# ---------------------------------------------------------------------------
# 1b. Repository preflight. Cheap, and it catches the class of failure that
#     wasted the last Stage C session.
# ---------------------------------------------------------------------------
import subprocess, sys

r = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_repo_layout.py", "-q",
     "--no-header", "-x"],
    capture_output=True, text=True, env={**__import__("os").environ, "PYTHONPATH": "src"},
)
print(r.stdout[-3000:])
if r.returncode != 0:
    raise SystemExit(
        "Repository layout is broken — see the assertion above.\n"
        "Do NOT continue: every train/evaluate call in this notebook will fail\n"
        "after the expensive tokenisation cells have already run."
    )
print("preflight OK")

In [ ]:
# ---------------------------------------------------------------------------
# 2. Paths. Set STAGE_A_DS to your Stage A dataset directory.
# ---------------------------------------------------------------------------
from pathlib import Path

STAGE_A_DS = Path("/kaggle/input/datasets/dulharakaushalya/tc-wpn-stage-a-data")
STAGE_A = None
for cand in (STAGE_A_DS / "data" / "clean", STAGE_A_DS / "clean", STAGE_A_DS):
    if (cand / "pkl").exists():
        STAGE_A = cand
        break
if STAGE_A is None:
    raise SystemExit(f"no pkl/ under {STAGE_A_DS}")

COHORT_CSV = STAGE_A / "cohort_psych_mimic4idx.csv"
PLAN_DIR   = str(STAGE_A / "plans")
STEM, K, SEED = "psych_mimic4idx", 5, 42
RESULTS = "/kaggle/working/results"

# Blinded pkls must be WRITTEN, so they cannot live in the read-only input.
# Copy the unblinded pkls into working and build the blinded ones beside them,
# so every variant sits in one directory and shares the frozen plans.
PKL_DIR = "/kaggle/working/pkl"
!mkdir -p {PKL_DIR}
!cp {STAGE_A}/pkl/*.pkl {PKL_DIR}/
print("cohort:", COHORT_CSV, COHORT_CSV.exists())
!ls -la {PKL_DIR}

## 3. Build the blinded variants — with the names the code actually accepts

Two arms, chosen to separate two different questions:

- **`anxiety`** — removes anxiety/panic/phobia diagnosis vocabulary only.
  Answers: can the model detect anxiety without the word?
- **`anx_meds`** — that plus anxiolytic and antidepressant drug names. This is
  your main robustness arm, and it is what you should call *strict lexical
  blinding* in the paper. Do **not** call it `strict_blind` without saying what
  it removes.

`psych` also exists (adds broad psychiatric vocabulary), but note the audit found
`other_psych` terms at a ratio of 0.83 — **more** common in controls than cases —
so blinding them removes a control-favouring cue, not a case-favouring one. That
makes `psych` a different experiment, not a stricter version of the same one.

One caveat to record: `tokenize_cohort.py`'s `ANXIETY_TERMS` does not include the
worry/nervous/restless family, and its `MED_TERMS` omits trazodone, mirtazapine,
propranolol, midazolam and chlordiazepoxide. So this is **diagnosis-and-drug-name
blinding**, not symptom blinding. State that scope explicitly.

In [ ]:
COHORT = str(COHORT_CSV)
!python -m scripts.tokenize_cohort --cohort {COHORT} --out {PKL_DIR} --blind anxiety
!python -m scripts.tokenize_cohort --cohort {COHORT} --out {PKL_DIR} --blind anx_meds
!ls {PKL_DIR}

In [ ]:
# ---------------------------------------------------------------------------
# 4. Verify the blinded pkls pair with the frozen plans BEFORE evaluating.
#    evaluate.py checks this too, but failing here costs seconds instead of
#    eight minutes of T4 time.
# ---------------------------------------------------------------------------
import sys, json
sys.path.insert(0, "/kaggle/working/tcwpn_test/src")
from tcwpn.sampler import RecordStore, EpisodePlan, store_fingerprint

for suffix in ("", "_blind-anxiety", "_blind-anx_meds"):
    store = RecordStore.from_pkl(f"{PKL_DIR}/{STEM}_test{suffix}.pkl", "test")
    plan = EpisodePlan.load(f"{PLAN_DIR}/{STEM}_test_k{K}.json")
    fp, want = store_fingerprint(store), plan.meta.get("store_fingerprint")
    print(f"{suffix or '(unblinded)':>18}  n={len(store.records):>6}  "
          f"fingerprint {fp}  plan wants {want}  "
          f"{'MATCH' if fp == want else 'MISMATCH'}")

## 5. The ablation — this is the scientific point of the notebook

Four configs, same frozen episodes, same patients, same backbone, one seed. Read
them as a ladder:

| config | what it adds |
|---|---|
| `protonet` | nothing (your Stage B baseline, tau frozen at 10.0) |
| `protonet_temp` | learnable tau only |
| `temporal_only` | learnable tau + `w^T` |
| `pcw_only` | learnable tau + `w^C` |
| `tcwpn_full` | both + auxiliary head (already trained in Stage B) |

**A prediction worth writing down before you look.** Your cohort has a median of
1 note per patient, and the sampler draws one note per support patient, so `w^T`
weights *patients against each other* by how long before their own index the note
was written — it is not weighting a trajectory. With that little temporal
structure I expect `temporal_only` to land close to `protonet_temp`. If it does,
say so; the honest ablation is more valuable than a flattering one.

The runs now log `lambda` and `beta` alongside `tau`, so you can report what the
mechanisms learned rather than only that something improved.

In [ ]:
CONFIGS = ["protonet_temp", "temporal_only", "pcw_only"]
for cfg in CONFIGS:
    print("="*70, f"\nTRAINING {cfg}\n", "="*70, sep="")
    !python -m scripts.train --config configs/{cfg}.yaml \
        --k {K} --seed {SEED} --stem {STEM} \
        --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --results {RESULTS}

### Re-run the two Stage B configs here too

They were trained against pkls in the read-only input; retraining them in this
session against `{PKL_DIR}` keeps every row of the ablation table produced by one
code version, which is what removes the "did the baseline differ for some other
reason" objection. Keep the Stage B artefacts as your frozen reference and
compare the two — if `tcwpn_full` does not reproduce ≈0.733 here, something in
the environment moved and that is worth knowing before Phase 3.

In [ ]:
for cfg in ["protonet", "tcwpn_full"]:
    print("="*70, f"\nRETRAINING {cfg}\n", "="*70, sep="")
    !python -m scripts.train --config configs/{cfg}.yaml \
        --k {K} --seed {SEED} --stem {STEM} \
        --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --results {RESULTS}

In [ ]:
# ---------------------------------------------------------------------------
# 6. Evaluate every config on the unblinded test split
# ---------------------------------------------------------------------------
ALL = ["protonet", "protonet_temp", "temporal_only", "pcw_only", "tcwpn_full"]
for cfg in ALL:
    run = f"{RESULTS}/{STEM}/{cfg}_k{K}_seed{SEED}"
    !python -m scripts.evaluate --run {run} --split test \
        --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --bootstrap 2000

In [ ]:
# ---------------------------------------------------------------------------
# 7. Blinded evaluation -- TC-WPN and the strongest baseline only.
#    Same checkpoints, same patients, same episodes; only the text differs.
# ---------------------------------------------------------------------------
for cfg in ["tcwpn_full", "protonet_temp"]:
    run = f"{RESULTS}/{STEM}/{cfg}_k{K}_seed{SEED}"
    for level in ["anxiety", "anx_meds"]:
        !python -m scripts.evaluate --run {run} --split test --blind {level} \
            --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --bootstrap 2000

## 8. Baseline sanity check — do not skip this

In Stage B, TF-IDF, the ClinicalBERT probe and ProtoNet all reported
**sensitivity 0.9993, specificity 0.0000** and the identical `f1_macro` of
0.3725. Three different models producing byte-identical operating points means
all three predicted the positive class for essentially every patient, and their
AUROCs (0.49–0.53) were at chance.

That may be the honest truth about few-shot learning on this task. It may also
mean the scores are near-constant and the locked threshold is falling below the
whole distribution. Those two possibilities have very different write-ups, and
the spread of the predicted probabilities tells you which.

In [ ]:
import pandas as pd, glob, os
rows = []
for f in sorted(glob.glob(f"{RESULTS}/{STEM}/*/predictions_test.csv")):
    d = pd.read_csv(f)
    p = d["p_anxiety"]
    rows.append({
        "run": os.path.basename(os.path.dirname(f)),
        "p_min": round(p.min(), 4), "p_p05": round(p.quantile(.05), 4),
        "p_median": round(p.median(), 4), "p_p95": round(p.quantile(.95), 4),
        "p_max": round(p.max(), 4), "p_sd": round(p.std(), 4),
        "n_unique_p": p.round(6).nunique(),
        "mean_p_case": round(p[d.label == 1].mean(), 4),
        "mean_p_control": round(p[d.label == 0].mean(), 4),
    })
if not rows:
    print("No predictions_test.csv found under", f"{RESULTS}/{STEM}/")
    print("Nothing evaluated yet — the training or evaluation cells above did")
    print("not complete. Scroll up and read the first error, not the last.")
    diag = pd.DataFrame()
else:
    diag = pd.DataFrame(rows).set_index("run")
    print(diag.to_string())
print()
print("Read it this way:")
print("  p_sd near 0 or n_unique_p tiny -> the model is emitting a near-constant")
print("     score. AUROC at chance then means 'learned nothing', and the")
print("     degenerate threshold is a symptom, not a separate bug.")
print("  p_sd healthy but mean_p_case ~= mean_p_control -> the model separates")
print("     nothing despite spread; still chance, but a different story.")
print("  mean_p_case > mean_p_control while AUROC ~ 0.5 -> check the positive")
print("     column orientation before writing anything down.")
if not diag.empty:
    diag.to_csv("/kaggle/working/baseline_score_diagnostics.csv")

## 9. Ablation and robustness tables

In [ ]:
!python -m scripts.compare_models table --results {RESULTS}/{STEM} --split test \
    --out /kaggle/working/ablation_table.csv

In [ ]:
# Paired DeLong along the ablation ladder. Each pair isolates ONE switch.
import itertools, subprocess, json
PAIRS = [
    ("protonet",      "protonet_temp",  "effect of learnable temperature alone"),
    ("protonet_temp", "temporal_only",  "effect of w^T, given tau"),
    ("protonet_temp", "pcw_only",       "effect of w^C, given tau"),
    ("pcw_only",      "tcwpn_full",     "what the rest of TC-WPN adds over w^C"),
    ("protonet_temp", "tcwpn_full",     "TC-WPN vs the FAIR baseline"),
    ("protonet",      "tcwpn_full",     "Stage B headline, for continuity"),
]
out = []
for a, b, why in PAIRS:
    pa = f"{RESULTS}/{STEM}/{a}_k{K}_seed{SEED}/predictions_test.csv"
    pb = f"{RESULTS}/{STEM}/{b}_k{K}_seed{SEED}/predictions_test.csv"
    dst = f"/kaggle/working/delong_{a}_vs_{b}.json"
    subprocess.run(["python", "-m", "scripts.compare_models", "pair",
                    "--a", pa, "--b", pb, "--out", dst], check=False)
    try:
        d = json.load(open(dst)); d["question"] = why; out.append(d)
    except Exception as e:
        print("skipped", a, b, e)

import pandas as pd
if out:
    df = pd.DataFrame(out)[["question", "auroc_a", "auroc_b", "delta_auroc",
                            "p_value", "n_patients"]]
    print(df.to_string(index=False))
    df.to_csv("/kaggle/working/delong_ladder.csv", index=False)
print()
print("Six tests on one test set: apply Holm correction before claiming")
print("significance. At alpha=0.05 the smallest p must beat 0.05/6 = 0.0083.")

In [ ]:
# ---------------------------------------------------------------------------
# 10. Robustness: retention of AUROC under blinding
# ---------------------------------------------------------------------------
import json, glob, os
import pandas as pd

rows = []
for run in sorted(glob.glob(f"{RESULTS}/{STEM}/*")):
    name = os.path.basename(run)
    for tag, label in [("test", "original"),
                       ("test_blind-anxiety", "anxiety-blind"),
                       ("test_blind-anx_meds", "anxiety+med-blind")]:
        f = os.path.join(run, f"eval_{tag}.json")
        if not os.path.exists(f):
            continue
        m = json.load(open(f))["metrics"]
        rows.append({"run": name, "condition": label, "auroc": m["auroc"],
                     "ci_low": m["auroc_ci_lower"], "ci_high": m["auroc_ci_upper"],
                     "pr_auc": m["pr_auc"], "ece": m["ece"]})

if rows:
    r = pd.DataFrame(rows)
    piv = r.pivot(index="run", columns="condition", values="auroc")
    if "original" in piv.columns:
        for c in piv.columns:
            if c != "original":
                piv[f"retention_{c}"] = (piv[c] / piv["original"]).round(3)
    print(piv.to_string())
    r.to_csv("/kaggle/working/robustness_table.csv", index=False)
    print()
    print("Chance is 0.5, so read retention against the ABOVE-CHANCE margin,")
    print("not the raw ratio: an AUROC falling 0.73 -> 0.62 keeps 85% of the")
    print("raw number but loses nearly half the signal (0.23 -> 0.12).")
else:
    print("no eval files found")

## 11. What to send your supervisor

1. `ablation_table.csv` — the five-row ladder
2. `delong_ladder.csv` — six paired tests, Holm threshold noted
3. `robustness_table.csv` — original vs two blinding levels
4. `baseline_score_diagnostics.csv` — whether the at-chance baselines were
   degenerate or genuinely uninformative
5. The learned `lambda` and `beta` from the training logs

The claim to make is narrow and defensible:

> On the patient-disjoint MIMIC-IV test cohort (2,278 patients, 59.4% cases),
> TC-WPN achieved AUROC 0.7335 (95% bootstrap CI 0.7135–0.7543) versus X for the
> temperature-matched ProtoNet baseline. This is a single-seed K=5 result.

Not "73% accuracy". Not "robust to lexical shortcuts" unless section 10 says so.
And describe the task as **concurrent detection at or before the index
admission**, not forecasting — the `at_or_before` policy is what built this
cohort.

Only after this lands should you spend quota on Phase 3 (K sweep) and Phase 4
(seeds).